# درس 6: آزمون فرضیه و p-value

متن کامل درس: [https://quantacademy.ir/courses/quant-math/hypothesis-testing-p-values/](https://quantacademy.ir/courses/quant-math/hypothesis-testing-p-values/)

داده‌های این دوره **شبیه‌سازی‌شده** هستند، نه قیمت واقعی بازار؛ هدف یاد دادن روش است. آموزشی است و توصیه سرمایه‌گذاری نیست.

سلول‌ها را از بالا به پایین اجرا کن (Shift+Enter).

## ۱. فرضیه صفر و آماره t

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

URL = "https://raw.githubusercontent.com/quantacademyir/quant-math/main/data/btc_daily_5y.csv"
close = pd.read_csv(URL, parse_dates=["date"], index_col="date")["close"]
r = np.log(close).diff().dropna()
n = len(r)

t_stat = r.mean() / (r.std(ddof=1) / np.sqrt(n))
p_value = 2 * stats.t.sf(abs(t_stat), df=n - 1)
print(round(t_stat, 3), round(p_value, 3))
res = stats.ttest_1samp(r, 0.0)
print(round(res.statistic, 3), round(res.pvalue, 3))

## ۲. p-value را شبیه‌سازی کن

In [ ]:
rng = np.random.default_rng(11)
sims = rng.normal(0, r.std(ddof=1), size=(20_000, n))
t_sims = sims.mean(axis=1) / (sims.std(axis=1, ddof=1) / np.sqrt(n))
print(round((np.abs(t_sims) >= abs(t_stat)).mean(), 3))
print(round((np.abs(t_sims) >= 1.96).mean(), 3))

## ۳. دو خطا و توان آزمون

In [ ]:
def rejection_rate(true_sharpe, n_days, trials=4000, seed=0):
    rng = np.random.default_rng(seed)
    mu = true_sharpe / np.sqrt(365) * 0.03
    x = mu + 0.03 * rng.standard_normal(size=(trials, n_days))
    t = x.mean(axis=1) / (x.std(axis=1, ddof=1) / np.sqrt(n_days))
    return (np.abs(t) > 1.96).mean()

for sr in [0, 0.25, 0.5, 1.0, 2.0]:
    print(sr, round(rejection_rate(sr, n), 3))

In [ ]:
for years in [1, 2, 5, 10, 20]:
    print(years, round(rejection_rate(0.5, 365 * years), 3))

## ۴. آزمون‌های دیگر روی همین داده

In [ ]:
rho = r.autocorr(1)
z = rho * np.sqrt(n)
print(round(rho, 3), round(z, 2), round(2 * stats.norm.sf(abs(z)), 3))

half = n // 2
w = stats.ttest_ind(r.iloc[:half], r.iloc[half:], equal_var=False)
print(round(r.iloc[:half].mean(), 5), round(r.iloc[half:].mean(), 5), round(w.pvalue, 3))

## ۵. p-value چه نیست؟

## ۶. وقتی چند آزمون می‌کنی

In [ ]:
rng = np.random.default_rng(12)
noise = rng.normal(0, 0.03, size=(100, n))
pv = stats.ttest_1samp(noise, 0.0, axis=1).pvalue
print((pv < 0.05).sum(), round(pv.min(), 4))
print((pv < 0.05 / 100).sum(), 0.05 / 100)

## ۷. معنادار آماری، بی‌ارزش عملی

In [ ]:
mu_true, sigma = 0.00002, 0.003
for n_obs in [1_825, 100_000, 10_000_000]:
    t = mu_true / (sigma / np.sqrt(n_obs))
    print(n_obs, round(t, 2), f"{2 * stats.norm.sf(t):.2g}")
print(mu_true / 0.0005)